In [ ]:
import matplotlib.pyplot as plt
import matplotlib
# matplotlib.use("TkAgg")
from sklearn.manifold import TSNE                   # final reduction
from umap import UMAP
import numpy as np                                  # array handling
import random
import pandas as pd
import requests
import gensim.models
from tqdm import tqdm
import json

np.random.seed(0)
random.seed(0)

In [ ]:

# domain = "book"
domain = "movie"
# domain = "song"

to_unique = {}
unique_to_genre = {}
v = 0
invented = 0

h = 0


In [ ]:

def reduce_dimensions(model):
    num_dimensions = 2  # final num dimensions (2D, 3D, etc)

    # extract the words & their vectors, as numpy arrays
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

    # reduce using t-SNE
    # tsne = TSNE(n_components=num_dimensions, random_state=0, early_exaggeration=12, perplexity=30)
    tsne = UMAP()
    vectors = tsne.fit_transform(vectors)

    # pca = IncrementalPCA(n_components=num_dimensions, whiten=True)
    # vectors = pca.fit_transform(vectors)

    x_vals = [v[0] for v in vectors]
    y_vals = [v[1] for v in vectors]
    return x_vals, y_vals, labels

def plot_with_matplotlib(x_vals, y_vals, x_vals_=[],y_vals_=[], labels=[], colors=[]):
    if len(labels):
        df = pd.DataFrame.from_dict({
            "x": x_vals,
            "y": y_vals,
            "labels": labels,
            }
        )
    else:
        df = pd.DataFrame.from_dict({
            "x": x_vals,
            "y": y_vals,
            }
        )
    if len(colors):
        if len(x_vals_) and len(y_vals_):     
            plt.scatter(x_vals_, y_vals_,s=1, c="grey", alpha=0.01)

        plt.scatter(df['x'], df['y'],s=1, c=colors, alpha=1)
    else:
        plt.scatter(df['x'], df['y'],s=1)

    if len(labels):

        indices = list(range(len(labels)))
        selected_indices = random.sample(indices, 300)
        for i in selected_indices:
            plt.annotate(labels[i], (x_vals[i], y_vals[i]), fontsize=3)

    # plt.show()


In [ ]:
try:
    with open(f"{domain}_8.txt", "r", encoding="utf8") as f:
        corpus = f.read()
except:
    with open(f"{domain}_7.txt", "r", encoding="utf8") as f:
        corpus = f.read()

sentence_list = [lst.replace(" ", "_").replace("&", "and") for lst in corpus.lower().split("\n")]
print(sentence_list[0])

sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]

print(sentence_list[0])

outputs = {}
for i in sentence_list:
    individuals = i.split("//")

    individuals = [x.replace(" ", "_").replace("&", "and").lower() for x in individuals]


    s = individuals[0]
    e = [x for x in individuals if len(x)]


    if not len(e):
        continue

    if s == e[0]:
        individuals = [x[1:] if (x[0] == "_") else x for x in individuals]

    else:
        continue

    outputs[individuals[0]] = individuals[1:]


print(list(outputs.keys())[:5])

print("\n\nmeta")
try:
    with open(f"meta/{domain}_meta_fullv8_v3.json", "r", encoding="utf8") as f:
        corpus = json.load(f)
except:
    try:
        with open(f"meta/{domain}_meta_full.json", "r", encoding="utf8") as f:
            corpus = json.load(f)
    except:
        with open(f"meta/{domain}_meta.json", "r", encoding="utf8") as f:
            corpus = json.load(f)
c_keys = list(corpus.keys())
# c_keys = [lst.replace(" ", "_").replace("&", "and").lower() for lst in c_keys]
# c_keys = [lst[1:] if (lst[0] == "_")  else lst for lst in c_keys ]

# sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]
c = 0
b = 0
for i in tqdm(c_keys):
    i_ = i.replace(" ", "_").replace("&", "and").lower()
    i_ = i_ if i_[0] != "_" else i_[1:]
    if (i_ in list(outputs.keys())):
        # print(outputs[i_])
        # print(outputs[i])

        if type(corpus[i]) is dict:


            corpus[i]["similar"] = outputs[i_]
            c +=1
    else:
        if b < 11:
            print(i)
            b += 1
#     else:
#         print("works")
    
#     if c == 10:
#         break


In [ ]:

entity_list = []


recreated = []
c_ = 0
for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        if "similar" in list(corpus[i].keys()):
            lst = ""
            s = corpus[i]["similar"]
            i_ = i.replace(" ", "_").replace("&", "and").lower()
            i_ = i_ if i_[0] != "_" else i_[1:]
            lst = i_
            for j in s:
                lst += f"//{j}"
            c_+=1
            recreated.append(lst)

print(c_)
# assert False
for sent in sentence_list:
    entity_list += sent.split("//")
entity_list = sorted(list(set(entity_list)))

print(f"Corpus is {len(sentence_list)} sentences long")
print(f"There are {len(entity_list)} unique entities...")


num_perms = 5
permed_sentence_list = []
for sent in tqdm(sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)
#model.wv.save(f"books.wordvectors"


In [ ]:

x_vals, y_vals, labels = reduce_dimensions(model)


In [ ]:

# plot_with_matplotlib(x_vals, y_vals, labels=[])
# plt.show()
# assert False


In [ ]:
c_ = 0
unique_t_c_d = []
unique_t_c = []
unique_c = []
v = 0
invented = 0
for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        v += 1

        # if "similar" in corpus[i].keys():
        #     if len(corpus[i]["similar"]):
        if ("title" in corpus[i].keys()) and ("cast" in (corpus[i].keys())) and ("directors" in (corpus[i].keys())):
            # if (corpus[i]["cast"] is not None):

                cast = ""
                if corpus[i]["cast"] is None:
                    cast = "NONE "
                else:
                    for ca in corpus[i]["cast"]:
                        # print(ca)
                        cast += f"{ca} "
                
                directors = ""
                if corpus[i]["directors"] is None:
                    directors = "NONE "
                else:
                    for di in corpus[i]["directors"]:
                        # print(ca)
                        directors += f"{di} "
                # print(cast)
                unique_t_c.append(corpus[i]["title"]+cast)
                unique_t_c_d.append(corpus[i]["title"]+cast+directors)

                unique_c.append(cast)
                c_ += 1
        # assert False
    else:
        if "invented" in corpus[i].lower():
            invented += 1

print(c_)
print("title+cast+dir:",len(set(unique_t_c_d)))
print("title+cast:",len(set(unique_t_c)))

print("cast:",len(set(unique_c)))


print(v)
print(v/(1.0*len(c_keys)))
print(invented/(1.0*len(c_keys)))


uni_id = []

for i in list(to_unique.keys()):
    if to_unique[i] not in uni_id:
        uni_id.append(to_unique[i])

print(len(set(uni_id)))

In [ ]:
to_unique = {}
unique_to_genre = {}


for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        v += 1

        # if "similar" in corpus[i].keys():
        #     if len(corpus[i]["similar"]):
        if ("title" in corpus[i].keys()) and ("cast" in (corpus[i].keys())) and ("directors" in (corpus[i].keys())):
            # if (corpus[i]["cast"] is not None):

                cast = ""
                if corpus[i]["cast"] is None:
                    cast = "NONE "
                else:
                    for ca in corpus[i]["cast"]:
                        # print(ca)
                        cast += f"{ca} "
                
                directors = ""
                if corpus[i]["directors"] is None:
                    directors = "NONE "
                else:
                    for di in corpus[i]["directors"]:
                        # print(ca)
                        directors += f"{di} "
                # print(cast)
                i_ = i.replace(" ", "_").replace("&", "and").lower()
                i_ = i_ if i_[0] != "_" else i_[1:]
                id_ = (corpus[i]["title"]+cast).replace(" ", "_")
                to_unique[i_] = id_
                if corpus[i]["genres"] is not None:
                    unique_to_genre[id_] = corpus[i]["genres"]
                else:
                    unique_to_genre[id_] = None
                # unique_t_c_d.append(corpus[i]["title"]+cast+directors)

                # unique_c.append(cast)

In [ ]:
num_perms = 5
permed_sentence_list = []
tot = 0
skipped = 0
new_sentence_list = []

# print(len(sentence_list))

# print(sentence_list[:5])

for sent in tqdm(sentence_list):
    ents = sent.split("//")
    ent = ""
    try:
        to_unique[ents[0]]
    except:
        # print(ents[0])
        continue
    for e in ents:
        try:
            ent += f"{to_unique[e]}//"
        except:
            skipped += 1
            # print(e)
            # assert False
            continue
    
    ent = ent[:-2]

    new_sentence_list.append(ent)
    tot+=1

print(tot)
print(skipped)

# assert False


for sent in tqdm(new_sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)


In [ ]:

#model.wv.save(f"books.wordvectors"

In [ ]:

# plot_with_matplotlib(x_vals, y_vals, labels=[])
# plt.show()

In [ ]:
total_uniques = len(to_unique.keys())


uniques = []
for i in tqdm(list(to_unique.keys())):
    if to_unique[i] not in uniques:
        uniques.append(to_unique[i])

print(len(uniques))



In [ ]:
# model.wv.save(f"movie.wordvectors")
# assert False

In [ ]:
import seaborn as sns

genre_colors = {}
genres = [unique_to_genre[x] for x in labels]

unique_genres = []

for lst in genres:
    if type(lst) is list:
        for l in lst:
            if l not in unique_genres:
                unique_genres.append(l)

unique_genres = set(unique_genres)

print(unique_genres)

colors = sns.color_palette(None, len(unique_genres)).as_hex()

for i_g,g in enumerate(unique_genres):
    genre_colors[g] = colors[i_g]

print(genre_colors)

In [ ]:

def reduce_dimensions(model):
    num_dimensions = 2  # final num dimensions (2D, 3D, etc)

    # extract the words & their vectors, as numpy arrays
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

    # reduce using t-SNE
    tsne = TSNE(n_components=num_dimensions, random_state=0, early_exaggeration=12, perplexity=30)
    # tsne = UMAP(min_dist=0.99,n_neighbors=5)
    vectors = tsne.fit_transform(vectors)

    # pca = IncrementalPCA(n_components=num_dimensions, whiten=True)
    # vectors = pca.fit_transform(vectors)

    x_vals = [v[0] for v in vectors]
    y_vals = [v[1] for v in vectors]
    return x_vals, y_vals, labels

In [ ]:
x_vals, y_vals, labels = reduce_dimensions(model)
anim_fam_idx = []

for idx_i, i in enumerate(labels):

    if unique_to_genre[i] is None:
        continue
    elif ("Family" in unique_to_genre[i]) and ("Animation" in unique_to_genre[i]) and len(unique_to_genre[i]) == 2:
        anim_fam_idx.append(idx_i)

horror_thrill_idx = []

for idx_i, i in enumerate(labels):

    if unique_to_genre[i] is None:
        continue
    elif ("Horror" in unique_to_genre[i]) and ("Thriller" in unique_to_genre[i]) and len(unique_to_genre[i]) == 2:
        horror_thrill_idx.append(idx_i)


romcom_idx = []

for idx_i, i in enumerate(labels):

    if unique_to_genre[i] is None:
        continue
    elif ("Romance" in unique_to_genre[i]) and ("Comedy" in unique_to_genre[i]) and len(unique_to_genre[i]) == 2:
        romcom_idx.append(idx_i)

num_dimensions = 2  # final num dimensions (2D, 3D, etc)

# extract the words & their vectors, as numpy arrays
vectors = np.asarray(model.wv.vectors)
labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

num_dimensions = 2  # final num dimensions (2D, 3D, etc)

# extract the words & their vectors, as numpy arrays
vectors = np.asarray(model.wv.vectors)
labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

# reduce using t-SNE
tsne = TSNE(n_components=num_dimensions, random_state=0, early_exaggeration=12, perplexity=30)
# tsne = UMAP(min_dist=0.99,n_neighbors=5)
vectors = tsne.fit_transform(vectors)

# pca = IncrementalPCA(n_components=num_dimensions, whiten=True)
# vectors = pca.fit_transform(vectors)

# x_vals = [v[0] for v in vectors]
# y_vals = [v[1] for v in vectors]
# return x_vals, y_vals, labels

from operator import itemgetter



x_vals = [v[0] for v in vectors]
y_vals = [v[1] for v in vectors]

cols = ["#2288ff" for x in anim_fam_idx]
x_v = itemgetter(*anim_fam_idx)(x_vals)
y_v = itemgetter(*anim_fam_idx)(y_vals)
g_v = itemgetter(*anim_fam_idx)(genres)

plot_with_matplotlib(x_v, y_v,x_vals,y_vals, colors=cols)

cols = ["#ff0000" for x in horror_thrill_idx]
x_v = itemgetter(*horror_thrill_idx)(x_vals)
y_v = itemgetter(*horror_thrill_idx)(y_vals)
g_v = itemgetter(*horror_thrill_idx)(genres)

plot_with_matplotlib(x_v, y_v, colors=cols)

plt.title("Compare")
plt.savefig(f"figs/FamilyAnim-Horror.png", dpi=100)
plt.show()
plt.clf()


cols = ["#ff8899" for x in romcom_idx]
x_v = itemgetter(*romcom_idx)(x_vals)
y_v = itemgetter(*romcom_idx)(y_vals)
g_v = itemgetter(*romcom_idx)(genres)

plot_with_matplotlib(x_v, y_v,x_vals,y_vals, colors=cols)

cols = ["#000000" for x in horror_thrill_idx]
x_v = itemgetter(*horror_thrill_idx)(x_vals)
y_v = itemgetter(*horror_thrill_idx)(y_vals)
g_v = itemgetter(*horror_thrill_idx)(genres)

plot_with_matplotlib(x_v, y_v, colors=cols)

plt.title("Compare")
plt.savefig(f"figs/RomCom-Horror.png", dpi=100)
plt.show()
plt.clf()

assert False

for g in tqdm(unique_genres):
    idx = []
    for idx_i, i in enumerate(labels):
        if unique_to_genre[i] is None:
            continue
        elif g in unique_to_genre[i]:
            idx.append(idx_i)
    cols = [genre_colors[g] for x in idx]
    x_v = itemgetter(*idx)(x_vals)
    y_v = itemgetter(*idx)(y_vals)
    g_v = itemgetter(*idx)(genres)

    print(len(cols))
    print(len(x_v))
    print(len(y_v))
    print(len(g_v))
    print("----")

    # g_c = itemgetter(*idx)(gen_colors)
    # print(len(x_vals),len(y_vals))
    plot_with_matplotlib(x_v, y_v,x_vals,y_vals, colors=cols)
    plt.title(g)
    plt.savefig(f"figs/{g}.png", dpi=100)
    plt.show()
    plt.clf()

assert False

In [ ]:
print(labels[:5])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import lightgbm as lgb
# extract the words & their vectors, as numpy arrays
model_vectors = np.asarray(model.wv.vectors)
model_labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

print(len(model_vectors))
print(len(model_labels))

x = model_vectors.copy()


In [ ]:

clfs = {}

for g in unique_genres:
    rf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)
    clfs[g] = rf


idx = np.arange(len(model_vectors))

random.shuffle(idx)
param = {'num_leaves': 70, 'objective': 'binary', 'metric':['f1','binary_logloss'], 'verbosity':-1,
         'learning_rate':0.003, 'is_unbalance':True}


num_round = 50000

for g in sorted(unique_genres):

    x = model_vectors.copy()

    y = []
    i_to_remove = []
    for idx_i, i in enumerate(model_labels):
        if unique_to_genre[i] is None:
            i_to_remove.append(idx_i)

    x = np.delete(x, i_to_remove, axis=0)
    for idx_i, i in enumerate(model_labels):
        if unique_to_genre[i] is None:
            continue
        elif g in unique_to_genre[i]:
            y.append(1)
        else:
            y.append(0)

    assert len(x) == len(y)

    split_t = int(0.8*len(x))
    split_v = int(0.9*len(x))


    idx_t = [i for i in idx if i < len(x)]

    x_train = x[idx_t[:split_t]]
    y_train = [y[n] for n in idx_t[:split_t]]
    train_data = lgb.Dataset(x_train, label=y_train)
    print("---------------")
    print(g)

    print(np.sum(y_train), " positives in train")

    x_val = x[idx_t[split_t:split_v]]
    y_val = [y[n] for n in idx_t[split_t:split_v]]
    val_data = lgb.Dataset(x_val, label=y_val)
    print(np.sum(y_val), " positives in val")


    x_test = x[idx_t[split_v:]]
    y_test = [y[n] for n in idx_t[split_v:]]
    test_data = lgb.Dataset(x_train, label=y_test)
    print(np.sum(y_test), " positives in test")
    
    bst = lgb.train(param, train_data, num_round, valid_sets=[val_data],callbacks=[lgb.early_stopping(stopping_rounds=10000)])

    # clfs[g].fit(x_train,y_train)

    preds = bst.predict(x_train)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0

    print("Train Acc: ",accuracy_score(y_train,preds))
    print("Train F1: ",f1_score(y_train,preds))

    
    preds = bst.predict(x_val)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Val Acc: ",accuracy_score(y_val,preds))
    print("Val F1: ",f1_score(y_val,preds))

    preds = bst.predict(x_test)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Test Acc: ",accuracy_score(y_test,preds))
    print("Test F1: ",f1_score(y_test,preds))
    print()

    # cols = [genre_colors[g] for x in idx]
    # x_v = itemgetter(*idx)(x_vals)
    # y = 




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model_vectors = np.asarray(model.wv.vectors)
model_labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

print(len(model_vectors))
print(len(model_labels))

x = model_vectors.copy()
y = []

for idx_i, i in enumerate(model_labels):
    y_temp = [0]*len(unique_genres)
    for g_idx,g in enumerate(unique_genres):
        if unique_to_genre[i] is None:
            y_temp[g_idx] = 0
        elif g in unique_to_genre[i]:
            y_temp[g_idx] = 1
        else:
            y_temp[g_idx] = 0
    
    y.append(y_temp)

assert len(x) == len(y), f"{len(x)} vs {len(y)}"

split = int(0.8*len(x))
x_train = x[:split]
y_train = y[:split]

x_val = x[split:]
y_val = y[split:]



In [ ]:
clf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)

clf.fit(x_train,y_train)
# clfs[g].fit(x_train,y_train)


In [ ]:
# from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay, classification_report

preds_tr = clf.predict(x_train)
print(np.array(y_train).shape)
print(preds_tr.shape)

print(" train: ",accuracy_score(y_train,preds_tr))

# cm = confusion_matrix(y_train, preds_tr, labels=unique_genres)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=unique_genres)
# disp.plot()
# plt.show()
# classification_report(y_train,preds_tr)
preds_val = clf.predict(x_val)

# cm = confusion_matrix(y_val, preds_val, labels=unique_genres)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=unique_genres)
# disp.plot()
# plt.show()

print(" val: ",accuracy_score(y_val,preds_val))
# classification_report(y_val,preds_val)

for g_idx,g in enumerate(unique_genres):
    print(g," train: ",accuracy_score(np.array(y_train)[:,g_idx],preds_tr[:,g_idx]))
    print(g," val: ",accuracy_score(np.array(y_val)[:,g_idx],preds_val[:,g_idx]))


    # cols = [genre_colors[g] for x in idx]
    # x_v = itemgetter(*idx)(x_vals)
    # y = 


In [ ]:
assert False

In [ ]:
unique_to_director = {}


for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        v += 1

        # if "similar" in corpus[i].keys():
        #     if len(corpus[i]["similar"]):
        if ("title" in corpus[i].keys()) and ("cast" in (corpus[i].keys())) and ("directors" in (corpus[i].keys())):
            # if (corpus[i]["cast"] is not None):

                cast = ""
                if corpus[i]["cast"] is None:
                    cast = "NONE "
                else:
                    for ca in corpus[i]["cast"]:
                        # print(ca)
                        cast += f"{ca} "
                
                directors = ""
                if corpus[i]["directors"] is None:
                    directors = "NONE "
                else:
                    for di in corpus[i]["directors"]:
                        # print(ca)
                        directors += f"{di} "
                # print(cast)
                i_ = i.replace(" ", "_").replace("&", "and").lower()
                i_ = i_ if i_[0] != "_" else i_[1:]
                id_ = (corpus[i]["title"]+cast).replace(" ", "_")

                if corpus[i]["languages"] is not None:
                    if len(corpus[i]["languages"]):
                        unique_to_director[id_] = corpus[i]["languages"]
                    else:
                        unique_to_director[id_] = None

                else:
                    unique_to_director[id_] = None

In [ ]:
directs = [unique_to_director[x] for x in labels]

# print(direct)

unique_directors = []

for lst in directs:
    if type(lst) is list:
        for l in lst:
            if l not in unique_directors:
                unique_directors.append(l)

# print(unique_directors.count("Quentin Tarantino"))
# assert False
# unique_directors = [d for d in unique_directors if unique_directors.count(d)>10]
unique_directors = set(unique_directors)


print(unique_directors)
print(len(unique_directors))


# colors = sns.color_palette(None, len(unique_genres)).as_hex()

# for i_g,g in enumerate(unique_genres):
#     genre_colors[g] = colors[i_g]

# print(genre_colors)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model_vectors = np.asarray(model.wv.vectors)
model_labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

print(len(model_vectors))
print(len(model_labels))

# clfs = {}

# for g in unique_genres:
#     rf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)
#     clfs[g] = rf



# for g in unique_genres:

#     x = model_vectors.copy()

y = []

for idx_i, i in enumerate(model_labels):
    # y_temp = [0]*len(unique_genres)
    if unique_to_director[i] is None:
        y.append(0)
    elif "English" in unique_to_director[i]:
        y.append(1)
    else:
        y.append(0)
print(sum(y)/len(y)*1.0)
unique, counts = np.unique(y, return_counts=True)

print(counts)

assert len(x) == len(y), f"{len(x)} vs {len(y)}"

split = int(0.8*len(x))
x_train = x[:split]
y_train = y[:split]

x_val = x[split:]
y_val = y[split:]



In [ ]:
clf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)

clf.fit(x_train,y_train)

In [ ]:
preds_tr = clf.predict(x_train)
print(np.array(y_train).shape)
print(preds_tr.shape)

print(" train: ",accuracy_score(y_train,preds_tr))

# cm = confusion_matrix(y_train, preds_tr, labels=unique_genres)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=unique_genres)
# disp.plot()
# plt.show()
# classification_report(y_train,preds_tr)
preds_val = clf.predict(x_val)

# cm = confusion_matrix(y_val, preds_val, labels=unique_genres)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=unique_genres)
# disp.plot()
# plt.show()

print(" val: ",accuracy_score(y_val,preds_val))

In [ ]:

with open("meta/movielens_meta_ID_FINAL.json", 'rb') as f:
    movielens = json.load(f)

print(len(movielens.keys()))
ids = 0
for k in list(movielens.keys()):
    if type(movielens[k]) is dict:
        if "ID" in movielens[k].keys():
            ids += 1

print(ids)

#### SONGS

In [ ]:
with open(f"song_7.txt", "r", encoding="utf8") as f:
    corpus = f.read()

sentence_list = [lst.replace(" ", "_").replace("&", "and") for lst in corpus.lower().split("\n")]
print(sentence_list[0])
print(len(sentence_list))

sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]

print(len(sentence_list))

print(sentence_list[0])

t = 0

outputs = {}
for i in sentence_list:
    individuals = i.split("//")

    individuals = [x.replace(" ", "_").replace("&", "and").lower() for x in individuals]

    

    s = individuals[0]
    e = [x for x in individuals if len(x)]

    if not len(e):
        continue

    if s == e[0]:
        individuals = [x[1:] if (x[0] == "_") else x for x in individuals]

    else:
        
        print(s,e)
        assert False

        continue

    outputs[individuals[0]] = individuals[1:]

print(len(outputs.keys()))
# print(list(outputs.keys())[:5])

print("\n\nmeta")
with open(f"meta/song_meta.json", "r", encoding="utf8") as f:
    corpus = json.load(f)

c_keys = list(corpus.keys())
# c_keys = [lst.replace(" ", "_").replace("&", "and").lower() for lst in c_keys]
# c_keys = [lst[1:] if (lst[0] == "_")  else lst for lst in c_keys ]

# sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]
c = 0
b = 0

print(len(c_keys))

for i in tqdm(c_keys):
    i_ = i.replace(" ", "_").replace("&", "and").lower()
    i_ = i_ if i_[0] != "_" else i_[1:]

    if (i_ in list(outputs.keys())):
        # print(outputs[i_])
        # print(outputs[i])

        if type(corpus[i]) is dict:


            corpus[i]["similar"] = outputs[i_]
            c +=1
        else:
            if b < 10:
                print(i)
            b += 1
    else:
        b += 1

        continue


print(c)
print(b)
# print(corpus["100 bad days|ajr(2019)"])

In [ ]:
print(corpus["(i can't help) falling in love with you|ub40(1993)"])

In [ ]:

entity_list = []


recreated = []
c_ = 0
for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        if "similar" in list(corpus[i].keys()):
            lst = ""
            s = corpus[i]["similar"]
            i_ = i.replace(" ", "_").replace("&", "and").lower()
            i_ = i_ if i_[0] != "_" else i_[1:]
            lst = i_
            for j in s:
                lst += f"//{j}"
            c_+=1
            recreated.append(lst)

print(c_)
# assert False
for sent in sentence_list:
    entity_list += sent.split("//")
entity_list = sorted(list(set(entity_list)))

print(f"Corpus is {len(sentence_list)} sentences long")
print(f"There are {len(entity_list)} unique entities...")


num_perms = 5
permed_sentence_list = []
for sent in tqdm(sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)
#model.wv.save(f"books.wordvectors"
x_vals, y_vals, labels = reduce_dimensions(model)

In [ ]:

plot_with_matplotlib(x_vals, y_vals, labels=[])
plt.show()

In [ ]:
to_unique = {}
unique_to_genre = {}
v = 0
invented = 0

h = 0

for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        v += 1

        # if "similar" in corpus[i].keys():
        #     if len(corpus[i]["similar"]):
        if ("track" in corpus[i].keys()) and ("artist" in (corpus[i].keys())):
            # if (corpus[i]["cast"] is not None):

                artist = ""
                if corpus[i]["artist"] is None:
                    artist = "NONE "
                else:
                    for ar in corpus[i]["artist"]:
                        # print(ca)
                        artist += f"{ar} "
                
                i_ = i.replace(" ", "_").replace("&", "and").lower()
                i_ = i_ if i_[0] != "_" else i_[1:]
                id_ = (corpus[i]["track"]+" "+artist).replace(" ", "_")

                to_unique[i_] = id_
                if corpus[i]["tags"] is not None:
                    unique_to_genre[id_] = corpus[i]["tags"][:3]
                    if h<20:
                        print(corpus[i]["tags"][:3])
                    h += 1
                else:
                    unique_to_genre[id_] = None
                # unique_t_c_d.append(corpus[i]["title"]+cast+directors)

                # unique_c.append(cast)

print(v)
print(v/(1.0*len(c_keys)))
print(h)

uni_id = []

for i in list(to_unique.keys()):
    if to_unique[i] not in uni_id:
        uni_id.append(to_unique[i])

print(len(set(uni_id)))

In [ ]:
num_perms = 5
permed_sentence_list = []
tot = 0
skipped = 0
new_sentence_list = []

print(len(sentence_list))
for sent in tqdm(sentence_list):
    ents = sent.split("//")
    ent = ""
    try:
        to_unique[ents[0]]
    except:
        # print(ents[0])
        continue
    for e in ents:
        try:
            ent += f"{to_unique[e]}//"
        except:
            skipped += 1
            # print(e)
            # assert False
            continue
    
    ent = ent[:-2]

    new_sentence_list.append(ent)
    tot+=1

print(tot)
print(skipped)

# assert False


for sent in tqdm(new_sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)
#model.wv.save(f"books.wordvectors"
x_vals, y_vals, labels = reduce_dimensions(model)

In [ ]:
plot_with_matplotlib(x_vals, y_vals, labels=[])
plt.show()

In [ ]:
genre_colors = {}
genres = [unique_to_genre[x] for x in labels]

unique_genres = []

for lst in tqdm(genres):
    if type(lst) is list:
        for l in lst:
            # if l not in unique_genres:
            unique_genres.append(l)

unique_genres = [x for x in unique_genres if ((unique_genres.count(x) > 500) and (not x.isdigit()))]

unique_genres = set(unique_genres)

print(unique_genres)
print(len(unique_genres))


colors = sns.color_palette(None, len(unique_genres)).as_hex()

for i_g,g in enumerate(unique_genres):
    genre_colors[g] = colors[i_g]

print(genre_colors)

In [ ]:
from operator import itemgetter

a = False
for g in tqdm(unique_genres):
    if g == "Hip-Hop":
        continue

    idx = []
    for idx_i, i in enumerate(labels):
        t = g
        if g == "hip hop":
            t = "Hip-Hop"

        if unique_to_genre[i] is None:
            continue
        elif (g in unique_to_genre[i]) or (t in unique_to_genre[i]):
            idx.append(idx_i)
        
    cols = [genre_colors[g] for x in idx]
    x_v = itemgetter(*idx)(x_vals)
    y_v = itemgetter(*idx)(y_vals)
    g_v = itemgetter(*idx)(genres)

    print(len(cols))
    print(len(x_v))
    print(len(y_v))
    print(len(g_v))
    print("----")

    # g_c = itemgetter(*idx)(gen_colors)

    plot_with_matplotlib(x_v, y_v,x_vals, y_vals, colors=cols)
    plt.title(g)
    plt.savefig(f"figs/songs_{g}.png", dpi=100)
    plt.show()
    plt.clf()


In [ ]:
model_vectors = np.asarray(model.wv.vectors)
model_labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

print(len(model_vectors))
print(len(model_labels))

clfs = {}

for g in unique_genres:
    rf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)
    clfs[g] = rf


idx = np.arange(len(model_vectors))

random.shuffle(idx)
param = {'num_leaves': 70, 'objective': 'binary', 'metric':['f1','binary_logloss'], 'verbosity':-1,
         'learning_rate':0.003, 'is_unbalance':True}


num_round = 50000

for g in sorted(unique_genres):
    if g == "Hip-Hop":
        continue
    
    x = model_vectors.copy()

    y = []
    i_to_remove = []
    for idx_i, i in enumerate(model_labels):
        if i not in list(unique_to_genre.keys()):
            i_to_remove.append(idx_i)
        elif unique_to_genre[i] is None:
            i_to_remove.append(idx_i)

    t = g
    if g == "hip hop":
        t = "Hip-Hop"


    x = np.delete(x, i_to_remove, axis=0)
    for idx_i, i in enumerate(model_labels):
        if unique_to_genre[i] is None:
            continue
        elif ((g in unique_to_genre[i]) and (t in unique_to_genre[i])):
            y.append(1)
        else:
            y.append(0)

    assert len(x) == len(y)

    split_t = int(0.8*len(x))
    split_v = int(0.9*len(x))


    idx_t = [i for i in idx if i < len(x)]

    x_train = x[idx_t[:split_t]]
    y_train = [y[n] for n in idx_t[:split_t]]
    train_data = lgb.Dataset(x_train, label=y_train)
    print("---------------")
    print(g)

    print(np.sum(y_train), " positives in train")

    x_val = x[idx_t[split_t:split_v]]
    y_val = [y[n] for n in idx_t[split_t:split_v]]
    val_data = lgb.Dataset(x_val, label=y_val)
    print(np.sum(y_val), " positives in val")


    x_test = x[idx_t[split_v:]]
    y_test = [y[n] for n in idx_t[split_v:]]
    test_data = lgb.Dataset(x_train, label=y_test)
    print(np.sum(y_test), " positives in test")
    
    bst = lgb.train(param, train_data, num_round, valid_sets=[val_data],callbacks=[lgb.early_stopping(stopping_rounds=10000)])

    # clfs[g].fit(x_train,y_train)

    preds = bst.predict(x_train)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0

    print("Train Acc: ",accuracy_score(y_train,preds))
    print("Train F1: ",f1_score(y_train,preds))

    
    preds = bst.predict(x_val)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Val Acc: ",accuracy_score(y_val,preds))
    print("Val F1: ",f1_score(y_val,preds))

    preds = bst.predict(x_test)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Test Acc: ",accuracy_score(y_test,preds))
    print("Test F1: ",f1_score(y_test,preds))
    print()

    # cols = [genre_colors[g] for x in idx]
    # x_v = itemgetter(*idx)(x_vals)
    # y = 


# Books

In [ ]:
with open(f"book_7.txt", "r", encoding="utf8") as f:
    corpus = f.read()

sentence_list = [lst.replace(" ", "_").replace("&", "and") for lst in corpus.lower().split("\n")]
print(sentence_list[0])
print(len(sentence_list))

sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]

print(len(sentence_list))

print(sentence_list[0])

t = 0

outputs = {}
for i in sentence_list:
    individuals = i.split("//")

    individuals = [x.replace(" ", "_").replace("&", "and").lower() for x in individuals]

    

    s = individuals[0]
    e = [x for x in individuals if len(x)]

    if not len(e):
        continue

    if s == e[0]:
        individuals = [x[1:] if (x[0] == "_") else x for x in individuals]

    else:
        
        print(s,e)
        assert False

        continue

    outputs[individuals[0]] = individuals[1:]

print(len(outputs.keys()))
# print(list(outputs.keys())[:5])

print("\n\nmeta")
with open(f"meta/book_meta_full.json", "r", encoding="utf8") as f:
    corpus = json.load(f)

c_keys = list(corpus.keys())
# c_keys = [lst.replace(" ", "_").replace("&", "and").lower() for lst in c_keys]
# c_keys = [lst[1:] if (lst[0] == "_")  else lst for lst in c_keys ]

# sentence_list = [e for e in sentence_list if "similar" not in "".join(e)]
c = 0
b = 0

print(len(c_keys))

for i in tqdm(c_keys):
    i_ = i.replace(" ", "_").replace("&", "and").lower()
    i_ = i_ if i_[0] != "_" else i_[1:]

    if (i_ in list(outputs.keys())):
        # print(outputs[i_])
        # print(outputs[i])

        if type(corpus[i]) is dict:


            corpus[i]["similar"] = outputs[i_]
            c +=1
        else:
            if b < 10:
                print(i)
            b += 1
    else:
        b += 1

        continue


print(c)
print(b)
# print(corpus["100 bad days|ajr(2019)"])

In [ ]:

entity_list = []


recreated = []
c_ = 0
for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        if "similar" in list(corpus[i].keys()):
            lst = ""
            s = corpus[i]["similar"]
            i_ = i.replace(" ", "_").replace("&", "and").lower()
            i_ = i_ if i_[0] != "_" else i_[1:]
            lst = i_
            for j in s:
                lst += f"//{j}"
            c_+=1
            recreated.append(lst)

print(c_)
# assert False
for sent in sentence_list:
    entity_list += sent.split("//")
entity_list = sorted(list(set(entity_list)))

print(f"Corpus is {len(sentence_list)} sentences long")
print(f"There are {len(entity_list)} unique entities...")


num_perms = 5
permed_sentence_list = []
for sent in tqdm(sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)
#model.wv.save(f"books.wordvectors"
x_vals, y_vals, labels = reduce_dimensions(model)

In [ ]:

plot_with_matplotlib(x_vals, y_vals, labels=[])
plt.show()

In [ ]:
to_unique = {}
unique_to_genre = {}
v = 0
invented = 0

h = 0

for i in tqdm(c_keys):
    if type(corpus[i]) is dict:
        v += 1

        # if "similar" in corpus[i].keys():
        #     if len(corpus[i]["similar"]):
        if ("title" in corpus[i].keys()) and ("authors" in (corpus[i].keys())):
            # if (corpus[i]["cast"] is not None):
                if corpus[i]["title"] is None:
                    
                    invented += 1
                    continue

                artist = ""
                if corpus[i]["authors"] is None:
                    artist = "NONE "
                else:
                    for ar in corpus[i]["authors"]:
                        # print(ca)
                        artist += f"{ar} "
                
                i_ = i.replace(" ", "_").replace("&", "and").lower()
                i_ = i_ if i_[0] != "_" else i_[1:]
                id_ = (corpus[i]["title"]+" "+artist).replace(" ", "_")

                to_unique[i_] = id_
                if corpus[i]["subjects"] is not None:
                    unique_to_genre[id_] = corpus[i]["subjects"][:3]
                    if h<20:
                        print(corpus[i]["subjects"][:3])
                    h += 1
                else:
                    unique_to_genre[id_] = None
                # unique_t_c_d.append(corpus[i]["title"]+cast+directors)

                # unique_c.append(cast)

print(v)
print(v/(1.0*len(c_keys)))
print(invented)
print(h)

uni_id = []

for i in list(to_unique.keys()):
    if to_unique[i] not in uni_id:
        uni_id.append(to_unique[i])

print(len(set(uni_id)))

In [ ]:
num_perms = 5
permed_sentence_list = []
tot = 0
skipped = 0
new_sentence_list = []

print(len(sentence_list))
for sent in tqdm(sentence_list):
    ents = sent.split("//")
    ent = ""
    try:
        to_unique[ents[0]]
    except:
        # print(ents[0])
        continue
    for e in ents:
        try:
            ent += f"{to_unique[e]}//"
        except:
            skipped += 1
            # print(e)
            # assert False
            continue
    
    ent = ent[:-2]

    new_sentence_list.append(ent)
    tot+=1

print(tot)
print(skipped)

# assert False


for sent in tqdm(new_sentence_list):
    ents = sent.split("//")
    for I in range(2,min(len(ents),11)):
        these_ents = ents[:I]
        for p_i in range(num_perms):
            p = random.sample(these_ents, len(these_ents))
            permed_sentence_list.append("//".join(p))

print(f"Permuted corpus is {len(permed_sentence_list)} sentences long")
class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __init__(self, sentence_corpus, seperator="//"):
        self._corpus = sentence_corpus
        self._sep = seperator

    def __iter__(self):
        for line in self._corpus:
            ret = line.split(self._sep)
            yield ret

sentences = MyCorpus(permed_sentence_list)
model = gensim.models.Word2Vec(sentences=sentences, min_count=10, vector_size=100, window=6)
#model.wv.save(f"books.wordvectors"
x_vals, y_vals, labels = reduce_dimensions(model)

In [ ]:
plot_with_matplotlib(x_vals, y_vals, labels=[])
plt.show()

In [ ]:
genre_colors = {}
genres = [unique_to_genre[x] for x in labels if x in list(unique_to_genre.keys())]

unique_genres = []

for lst in tqdm(genres):
    if type(lst) is list:
        for l in lst:
            # if l not in unique_genres:
            unique_genres.append(l)

unique_genres = [x for x in tqdm(unique_genres) if ((unique_genres.count(x) > 300) and (not x.isdigit()))]

unique_genres = set(unique_genres)

print(unique_genres)
print(len(unique_genres))


colors = sns.color_palette(None, len(unique_genres)).as_hex()

for i_g,g in enumerate(unique_genres):
    genre_colors[g] = colors[i_g]

print(genre_colors)

In [ ]:
from operator import itemgetter

a = False
for g in tqdm(unique_genres):


    idx = []
    for idx_i, i in enumerate(labels):


        # if i not in list(unique_to_genre.keys()):
        #     continue
        try:
            if (unique_to_genre[i] is None):
                continue
            if idx_i >= len(genres):
                continue
            elif (g in unique_to_genre[i]):
                idx.append(idx_i)
        except:
            continue
        
    cols = [genre_colors[g] for x in idx]
    x_v = itemgetter(*idx)(x_vals)
    y_v = itemgetter(*idx)(y_vals)
    try:
        g_v = itemgetter(*idx)(genres)
    except:
        print(idx)
        print(np.max(idx))
        print(len(genres))

    print(len(cols))
    print(len(x_v))
    print(len(y_v))
    print(len(g_v))
    print("----")

    # g_c = itemgetter(*idx)(gen_colors)

    plot_with_matplotlib(x_v, y_v,x_vals,y_vals, colors=cols)
    plt.title(g)
    plt.savefig(f"figs/books_{g}.png", dpi=100)
    plt.show()
    plt.clf()


In [ ]:
model_vectors = np.asarray(model.wv.vectors)
model_labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

print(len(model_vectors))
print(len(model_labels))

clfs = {}

for g in unique_genres:
    rf = RandomForestClassifier(n_estimators=1000,n_jobs=-1)
    clfs[g] = rf


idx = np.arange(len(model_vectors))

random.shuffle(idx)
param = {'num_leaves': 70, 'objective': 'binary', 'metric':['f1','binary_logloss'], 'verbosity':-1,
         'learning_rate':0.003, 'is_unbalance':True}


num_round = 50000

for g in sorted(unique_genres):
    
    x = model_vectors.copy()

    y = []
    i_to_remove = []
    for idx_i, i in enumerate(model_labels):
        try:
            if unique_to_genre[i] is None:
                i_to_remove.append(idx_i)
        except:
            i_to_remove.append(idx_i)

    x = np.delete(x, i_to_remove, axis=0)
    for idx_i, i in enumerate(model_labels):
        if idx_i in i_to_remove:
            continue
        if unique_to_genre[i] is None:
            continue
        elif ((g in unique_to_genre[i])):
            y.append(1)
        else:
            y.append(0)

    assert len(x) == len(y)

    split_t = int(0.8*len(x))
    split_v = int(0.9*len(x))


    idx_t = [i for i in idx if i < len(x)]

    x_train = x[idx_t[:split_t]]
    y_train = [y[n] for n in idx_t[:split_t]]
    train_data = lgb.Dataset(x_train, label=y_train)
    print("---------------")
    print(g)

    print(np.sum(y_train), " positives in train")

    x_val = x[idx_t[split_t:split_v]]
    y_val = [y[n] for n in idx_t[split_t:split_v]]
    val_data = lgb.Dataset(x_val, label=y_val)
    print(np.sum(y_val), " positives in val")


    x_test = x[idx_t[split_v:]]
    y_test = [y[n] for n in idx_t[split_v:]]
    test_data = lgb.Dataset(x_train, label=y_test)
    print(np.sum(y_test), " positives in test")
    
    bst = lgb.train(param, train_data, num_round, valid_sets=[val_data],callbacks=[lgb.early_stopping(stopping_rounds=10000)])

    # clfs[g].fit(x_train,y_train)

    preds = bst.predict(x_train)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0

    print("Train Acc: ",accuracy_score(y_train,preds))
    print("Train F1: ",f1_score(y_train,preds))

    
    preds = bst.predict(x_val)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Val Acc: ",accuracy_score(y_val,preds))
    print("Val F1: ",f1_score(y_val,preds))

    preds = bst.predict(x_test)
    preds[preds >= 0.5] = 1
    preds[preds < 0.5] = 0
    print("Test Acc: ",accuracy_score(y_test,preds))
    print("Test F1: ",f1_score(y_test,preds))
    print()

    # cols = [genre_colors[g] for x in idx]
    # x_v = itemgetter(*idx)(x_vals)
    # y = 


In [ ]:

## interactive search...
search_dict = {s: s.replace("_", " ").replace("|", " ").replace("(", "").replace(")", "").strip().split(" ") for s in model.wv.index_to_key}


sterm = input("Which entity would you like to search for?\n").lower()
if sterm in search_dict.keys():
    print(f"Most similar entities to {sterm}:")
    print("\n".join([f"{mst[0]}: {mst[1]}" for mst in model.wv.most_similar(positive=[sterm], topn=50)]))
    print("\n\n")
else:
    shared = {s: len(set(sterm.split(" ")) & set(search_dict[s])) for s in search_dict.keys()};
    sim_sterms = '\n'.join(sorted(shared, key=shared.get, reverse=True)[:10])
    print(f"No entity matching that string, most similar are...\n{sim_sterms}")

